# 05 사진 분류 나누기 · 합치기

1. **나누기**: `data/tunnel_photos/`의 사진을 2명에게 나눠 `사진분류_나눔_2명/분류자N/` 폴더와 zip을 만든다
   - 모든 사람이 **같은 사진 20장(공통)**도 함께 분류한다 → 사람끼리 기준이 맞는지 확인용
2. 팀원은 zip을 풀고 `사진분류_도구.html`로 분류 → `사진분류_이름.csv`를 보내준다
3. **합치기**: 받은 csv를 `data/사진분류_받은것/`에 넣고 실행 → 공통 20장 일치도 확인 → `data/사진분류.csv` 저장 (04 노트북이 이 파일을 읽음)

사진을 다시 받아 장수가 늘었으면 1번부터 다시 실행하면 된다.

In [4]:
import pandas as pd, numpy as np, shutil, os
from pathlib import Path

PEOPLE = 2        # 나눌 사람 수
COMMON = 20       # 모두가 같이 보는 공통 사진 수
SEED = 42

PHOTO = Path('data/tunnel_photos')
OUT = Path(f'사진분류_나눔_{PEOPLE}명')   # 사람 수별 새 폴더
photos = sorted(p.name for p in PHOTO.glob('*.jpg'))
print('사진', len(photos), '장')

사진 172 장


In [5]:
# 1. 나누기
rng = np.random.default_rng(SEED)
order = list(rng.permutation(photos))
common, rest = order[:COMMON], order[COMMON:]
groups = [rest[k::PEOPLE] for k in range(PEOPLE)]

shutil.rmtree(OUT, ignore_errors=True)   # 다시 나눌 때 예전 결과 지우기
OUT.mkdir(exist_ok=True)
rows = []
for k, g in enumerate(groups, 1):
    d = OUT / f'분류자{k}'; (d/'사진').mkdir(parents=True, exist_ok=True)
    for f in common + g:
        shutil.copy(PHOTO/f, d/'사진'/f)
        rows.append({'image_id': f[:-4], '분류자': k, '공통': f in common})
    shutil.copy('사진분류_도구.html', d/'사진분류_도구.html')
    shutil.copy(OUT.parent/'사진분류_방법.txt', d/'사진분류_방법.txt') if Path('사진분류_방법.txt').exists() else None
    shutil.make_archive(str(OUT/f'분류자{k}'), 'zip', d)
    print(f'분류자{k}: {len(common)+len(g)}장 (공통 {len(common)} + 개인 {len(g)})')
pd.DataFrame(rows).to_csv(OUT/'나눔표.csv', index=False, encoding='utf-8-sig')
print('zip 위치:', OUT.resolve())

분류자1: 96장 (공통 20 + 개인 76)
분류자2: 96장 (공통 20 + 개인 76)
zip 위치: C:\Users\최재혁\OneDrive\바탕 화면\머신러닝3조\sunsight_video\사진분류_나눔_2명


In [ ]:
# 2. 합치기 ※ 지금은 실행하지 마세요. 팀원 csv를 data/사진분류_받은것/ 에 넣은 뒤 실행
IN = Path('data/사진분류_받은것')
parts = []
for f in sorted(IN.glob('사진분류_*.csv')):
    t = pd.read_csv(f, encoding='utf-8-sig', dtype={'image_id': str})
    t['who'] = f.stem.replace('사진분류_', '')
    parts.append(t)
IN.mkdir(parents=True, exist_ok=True)
if not parts:
    raise SystemExit('아직 받은 csv가 없어요. 팀원 분류가 끝나면 data/사진분류_받은것/ 에 넣고 이 칸부터 실행하세요.')
allc = pd.concat(parts, ignore_index=True)
print(allc.groupby('who').size())

common_ids = pd.read_csv(OUT/'나눔표.csv', dtype={'image_id': str}).query('공통')['image_id'].unique()
cm = allc[allc.image_id.isin(common_ids)].pivot_table(index='image_id', columns='who', values='class', aggfunc='first')
agree = (cm.nunique(axis=1) == 1).mean()
print(f'공통 {len(cm)}장 중 모두 같은 답: {agree:.0%}')
display(cm[cm.nunique(axis=1) > 1])   # 의견이 갈린 사진 → 같이 보고 기준 맞추기

In [ ]:
# 3. 최종 분류 저장 (공통 사진은 다수결, 동률·'애매'는 '애매'로 남김)
def vote(s):
    v = s.value_counts()
    return v.index[0] if (len(v) == 1 or v.iloc[0] > v.iloc[1]) else '애매'
final = allc.groupby('image_id')['class'].apply(vote).reset_index()
final.to_csv('data/사진분류.csv', index=False, encoding='utf-8-sig')
print(final['class'].value_counts())
print('애매 →', (final['class']=='애매').sum(), '장은 형이 다시 보기')